# Physical projection: hyperspectral
Run 00_prepare first. Shared cell structure for both paper cases.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import h5py
import matplotlib.pyplot as plt
from copy import deepcopy
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "docs/source_manifest.json").is_file())
sys.path.insert(0, str(ROOT))
from library import phase_retrieval_universal as universal
from library import phase_retrieval_geometry as geometry
CACHE = ROOT / "data/hyperspectral/prepared.h5"
OUTPUT = ROOT / "results/hyperspectral.h5"
THICKNESS_NM = None  # Enter independently measured total magnetic-material thickness.
with h5py.File(CACHE) as handle:
    holograms=handle['holograms'][()]; mask_pixel=handle['mask_pixel'][()]
    support=handle['supportmask'][()]
    config=json.loads(handle.attrs['config_json'])
    records=json.loads(handle.attrs['records_json'])
state_labels=[r['state'] for r in records]
energy_labels=[r['energy_eV'] for r in records]
polarizations=[r['polarization'] for r in records]
material=universal.largest_support_component(support).astype(float)
saturated={}  # Supply independently known saturated state labels when available.
current=None
crop, binning=config['crop'],config['binning']


## 1. Validate inputs and metadata

In [ ]:
assert holograms.ndim==3 and mask_pixel.shape==holograms.shape
assert support.shape==holograms.shape[-2:] and material.shape==support.shape
assert len(state_labels)==len(energy_labels)==len(polarizations)==len(holograms)
assert np.all(np.isfinite(holograms)) and np.any(support)
assert set(saturated.values()) <= {-1.,1.}
illumination_labels=["beam0"]*len(holograms)
metadata=dict(state_labels=state_labels, energy_labels=energy_labels,
              polarization_coefficients=polarizations, illumination_labels=illumination_labels)
print("Observations",len(holograms),"energies",sorted(set(energy_labels)),"anchors",saturated)
fig,ax=plt.subplots(1,2)
ax[0].imshow(np.log1p(np.maximum(holograms[0],0))); ax[0].set_title('First detector image')
ax[1].imshow(support); ax[1].set_title('Source support')


## 2. Set the physical model and iteration schedule

In [ ]:
# Shared controls. Change these explicitly for a new dataset.
MODES=[1,2]
OUTER_ITERATIONS=20
INNER_ITERATIONS=50
PHYSICAL_ITERATIONS=50
recipe=universal.default_universal_phase_retrieval_recipe()
recipe.update(projection_model="physical_factorized", modes=MODES,
    crop=crop, binning=binning, roi=None,
    recenter_modal_supports=True, mode_support_center="image",
    material_thickness=material, fit_material_thickness=False,
    physical_projection_object_roi=True, physical_phase_reference=True,
    saturated_states=saturated or None,
    mode_initialization="support_fft", mode_initialization_seed=0,
    partial_coherence=False,
    inner_mode="ER", inner_Nit=INNER_ITERATIONS,
    outer_iterations=OUTER_ITERATIONS, physical_iterations=PHYSICAL_ITERATIONS,
    projection_every=len(holograms), projection_relaxation=1.,
    final_projection_relaxation=1., final_fourier_constraint=False,
    constrain_nonphysical_modes_common=True,
    projection_constraints_inside_support_only=True, zero_magnetization_outside_support=True,
    clip_magnetization=False, magnetization_trend="none", bound_magnetization_median=False,
    physical_spatial_weighting="exit_wave_intensity", random_seed=0,
    observation_workers=1, fft_workers=1)
# Final model fields satisfy the physical projection; detector agreement is measured below.
recipe.update(energy_values=np.asarray(sorted(set(energy_labels))),
    charge_spectral_constraint="kk", magnetic_spectral_constraint="kk")


## 3. Independent phase retrieval warmup

In [ ]:
warmup_recipe=deepcopy(recipe)
warmup_recipe.update(projection_model="none", outer_iterations=0,
    final_projection_relaxation=0., final_fourier_constraint=False,
    constrain_nonphysical_modes_common=False, fit_illumination_intensity=False,
    fit_aperture_illumination=False, align_global_phase=False)
_, warmup, _, warm_masks, warm_errors=universal.universal_phase_retrieval_algorithm(
    holograms, mask_pixel, support, **metadata, universal_recipe=warmup_recipe,
    phase_retrieval_kernel=universal.multimode_phase_retrieval_kernel)
# Snapshot makes later accidental changes detectable before continuation.
import hashlib
warmup_signature=hashlib.sha256(holograms.tobytes()+mask_pixel.tobytes()+support.tobytes()+
    json.dumps(metadata,sort_keys=True).encode()+repr((crop,binning,MODES)).encode()).hexdigest()
warmup=warmup.copy(); warmup.setflags(write=False)


## 4. Alternate physical projection and phase retrieval

In [ ]:
signature=hashlib.sha256(holograms.tobytes()+mask_pixel.tobytes()+support.tobytes()+
    json.dumps(metadata,sort_keys=True).encode()+repr((crop,binning,MODES)).encode()).hexdigest()
if signature != warmup_signature: raise RuntimeError("Inputs changed; rerun warmup")
physical_recipe=deepcopy(recipe)
# The API accepts starts on the source detector grid. Embed the cached retrieval fields.
expanded=np.repeat(np.repeat(warmup,binning,axis=-2),binning,axis=-1)
origin=tuple(crop+(n-2*crop-u)//2 for n,u in zip(holograms.shape[-2:],expanded.shape[-2:]))
starts=np.zeros((*warmup.shape[:-2],*holograms.shape[-2:]),dtype=warmup.dtype)
starts[...,origin[0]:origin[0]+expanded.shape[-2],origin[1]:origin[1]+expanded.shape[-1]]=expanded
del expanded
measured, excluded, support_used, recovered, grid=geometry.prepare(
    holograms,mask_pixel,support,dict(physical_recipe,Nmodes=1),starts)
np.testing.assert_allclose(recovered,warmup,rtol=1e-12,atol=0)
del recovered
fields, received, components, bsmasks, errors=universal.universal_phase_retrieval_algorithm(
    holograms,mask_pixel,support,**metadata, universal_recipe=physical_recipe,start_fields=starts,
    phase_retrieval_kernel=universal.multimode_phase_retrieval_kernel)
np.testing.assert_allclose(received,warmup,rtol=1e-12,atol=0)
del starts,received
print("Physical model residual",components.get('fit_residual_rms'))
print("Identifiability",components.get('identifiability', 'See model calibration documentation'))


## 5. Compare detector residuals and inspect maps

In [ ]:
# Compare every observation on valid detector pixels, using the sum over incoherent modes.
detector_errors={}
for label,stack in [('warmup',warmup),('physical',fields)]:
    prediction=np.sum(abs(stack)**2,axis=1) if stack.ndim==4 else abs(stack)**2
    values=[]
    for i in range(len(measured)):
        valid=~np.asarray(bsmasks[i],bool)
        denominator=np.linalg.norm(measured[i][valid])
        values.append(np.linalg.norm((prediction[i]-measured[i])[valid])/denominator if denominator else np.nan)
    detector_errors[label]=np.asarray(values)
fig,ax=plt.subplots()
for label,values in detector_errors.items(): ax.plot(values,'o-',label=label)
ax.set(xlabel='Observation in acquisition order',ylabel='Relative intensity residual'); ax.legend()
magnetization=np.asarray(components['magnetization'])
fig,ax=plt.subplots(); im=ax.imshow(magnetization[0],cmap='RdBu_r'); fig.colorbar(im,ax=ax)
ax.set_title('First fitted state: magnetization factor')


## 6. Extract the paper observables

In [ ]:
def calibrate_responses(charge, magnetic, magnetization, energies_ev, *,
                        cobalt_thickness_nm=None, normalize_magnetization=False,
                        propagation_sign=-1):
    charge=np.asarray(charge, dtype=complex).copy()
    magnetic=np.asarray(magnetic, dtype=complex).copy()
    m=np.asarray(magnetization, dtype=float).copy()
    energy=np.asarray(energies_ev, dtype=float)
    if charge.ndim != 1 or magnetic.shape != charge.shape or energy.shape != charge.shape:
        raise ValueError('Expected matching one-dimensional charge, magnetic and energy spectra')
    if m.ndim != 3:
        raise ValueError('Require a state-by-image magnetization stack')
    if normalize_magnetization:
        scale=float(np.max(np.abs(m)))
        if scale == 0:
            raise ValueError('Cannot normalize an all-zero magnetization map')
        m = m / scale
        magnetic = magnetic * scale
    result=dict(charge_response=charge, magnetic_response=magnetic,
                magnetization=m, energies_ev=energy.copy(), propagation_sign=int(propagation_sign))
    if cobalt_thickness_nm is not None:
        if not np.isfinite(cobalt_thickness_nm) or cobalt_thickness_nm <= 0:
            raise ValueError('Cobalt thickness must be positive and finite in nm, or None')
        kt=2*np.pi*energy/1239.8419843320026*float(cobalt_thickness_nm)
        result.update(
            cobalt_thickness_nm=float(cobalt_thickness_nm),
            charge_index_contrast=charge/(propagation_sign*1j*kt),
            magnetic_index_contrast=magnetic/(propagation_sign*1j*kt),
            charge_delta=-propagation_sign*charge.imag/kt,
            charge_beta=-charge.real/kt,
            magnetic_delta=-propagation_sign*magnetic.imag/kt,
            magnetic_beta=-magnetic.real/kt,
        )
    return result

energies=np.asarray(components['energy_names'])
calibration=calibrate_responses(components['charge_response'],components['magnetic_response'],
    magnetization,energies,cobalt_thickness_nm=THICKNESS_NM,normalize_magnetization=False)
fig,axes=plt.subplots(1,2)
if THICKNESS_NM is None:
    for ax,key in zip(axes,['charge_response','magnetic_response']):
        ax.plot(energies,calibration[key].real,'o-',label='real')
        ax.plot(energies,calibration[key].imag,'o-',label='imag'); ax.set_title(key); ax.legend()
    print('Log-response units. Set measured thickness and establish spectral/magnetic gauges before claiming absolute indices.')
else:
    for ax,prefix in zip(axes,['charge','magnetic']):
        ax.plot(energies,calibration[prefix+'_delta'],'o-',label='delta')
        ax.plot(energies,calibration[prefix+'_beta'],'o-',label='beta'); ax.set_title(prefix); ax.legend()
for ax in axes: ax.set_xlabel('Photon energy (eV)')
paper_analysis=calibration


## 7. Save data, settings, and the figure

In [ ]:
# Store arrays and nested numerical metadata directly; this small writer stays visible here.
def save_tree(group, values):
    for key,value in values.items():
        if isinstance(value,dict): save_tree(group.create_group(str(key)),value)
        elif value is None: group.attrs[str(key)]='None'
        else:
            array=np.asarray(value)
            if array.dtype.kind in 'biufc': group.create_dataset(str(key),data=array)
            else: group.attrs[str(key)]=json.dumps(value,default=lambda x: np.asarray(x).tolist())
OUTPUT.parent.mkdir(parents=True,exist_ok=True)
if OUTPUT.exists(): raise FileExistsError(f'Choose a new OUTPUT path: {OUTPUT}')
with h5py.File(OUTPUT,'x') as handle:
    for name,value in dict(fields=fields,warmup_fields=warmup,holograms=measured,
                           mask_pixel=bsmasks,supportmask=support_used).items():
        handle.create_dataset(name,data=value,compression='gzip')
    save_tree(handle.create_group('metadata'),metadata)
    save_tree(handle.create_group('preprocessing'),config)
    save_tree(handle.create_group('recipe'),physical_recipe)
    save_tree(handle.create_group('warmup_recipe'),warmup_recipe)
    save_tree(handle.create_group('components'),components)
    save_tree(handle.create_group('errors'),errors)
    save_tree(handle.create_group('detector_errors'),detector_errors)
    save_tree(handle.create_group('paper_analysis'),paper_analysis)
    handle.attrs['warmup_signature']=warmup_signature
fig.savefig(OUTPUT.with_suffix('.pdf'),bbox_inches='tight')
print(OUTPUT)
